In [1]:
# pip install xgboost

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("titanic.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df = df.drop(['PassengerId','Name', 'Ticket','Cabin', 'Embarked'], axis=1)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
0,0,3,male,22.0,1,0,7.2500
1,1,1,female,38.0,1,0,71.2833
2,1,3,female,26.0,0,0,7.9250
3,1,1,female,35.0,1,0,53.1000
4,0,3,male,35.0,0,0,8.0500


In [7]:
df.Sex = df['Sex'].replace({'male' : 1, 'female' : 0})

C:\Users\ABDULLAH AL MASUM\AppData\Local\Temp\ipykernel_25464\2295240412.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.Sex = df['Sex'].replace({'male' : 1, 'female' : 0})


In [8]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
0,0,3,1,22.0,1,0,7.2500
1,1,1,0,38.0,1,0,71.2833
2,1,3,0,26.0,0,0,7.9250
3,1,1,0,35.0,1,0,53.1000
4,0,3,1,35.0,0,0,8.0500


In [9]:
df.shape

(891, 7)

In [10]:
df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
dtype: int64

In [11]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [15]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.drop_duplicates(inplace = True)

In [16]:
X = df.drop('Sex', axis=1)
Y = df[['Sex']]

In [17]:
X.head()

,Survived,Pclass,Age,SibSp,Parch,Fare
0,0,3,22.0,1,0,7.2500
1,1,1,38.0,1,0,71.2833
2,1,3,26.0,0,0,7.9250
3,1,1,35.0,1,0,53.1000
4,0,3,35.0,0,0,8.0500


In [18]:
Y.head()

,Sex
0,1
1,0
2,0
3,0
4,1


In [19]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, random_state= 42, test_size=.2)

In [20]:
std = StandardScaler()

In [21]:
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

In [22]:
xgb = XGBClassifier()

In [23]:
model1 = xgb.fit(x_train, y_train)

In [24]:
model1.score(x_train, y_train)

0.9549114331723028

In [25]:
model1.score(x_test, y_test)

0.7948717948717948

In [26]:
y_pred = xgb.predict(x_test)

In [27]:
accuracy_score(y_test, y_pred)

0.7948717948717948

In [39]:
cm1 = confusion_matrix(y_test, y_pred)

In [50]:
param_grid = {
  "n_estimators" : [100, 200, 300, 400],
  "learning_rate" : [.1, .001, 1, .5],
  "max_depth" : [3,5,7,9],
  "subsample" : [.6,.7,.8],
  "colsample_bytree" : [.7,.8,.9],
  "objective" : ['binary:logistic']
}

In [51]:
from sklearn.model_selection import GridSearchCV

In [52]:
grid_CV = GridSearchCV(estimator=xgb, param_grid=param_grid, n_jobs=-1, verbose=10 )

In [60]:
grid_CV.best_params_

{'colsample_bytree': 0.9,
 'learning_rate': 0.1,
 'max_depth': 3,
 'n_estimators': 100,
 'objective': 'binary:logistic',
 'subsample': 0.7}

In [53]:
model2 = grid_CV.fit(x_train, y_train)

Fitting 5 folds for each of 576 candidates, totalling 2880 fits


In [54]:
model2.score(x_train, y_train)

0.8615136876006442

In [55]:
model2.score(x_test, y_test)

0.8205128205128205

In [56]:
y_pred2 = model2.predict(x_test)

In [57]:
cm2 = confusion_matrix(y_pred2, y_test)

In [58]:
accuracy_score(y_pred2, y_test)

0.8205128205128205

In [59]:
print("Before parameter tuning : \n", cm1)
print("After parameter tuning : \n", cm2)


Before parameter tuning : 
 [[44 12]
 [20 80]]
After parameter tuning : 
 [[45 17]
 [11 83]]
